In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_teddynote.tools.tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_teddynote.messages import stream_graph

# 1. API 키 정보 로드 및 LangSmith 로깅 설정
load_dotenv()
logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Multi_React_Agent")

# 2. 메모리 및 모델 설정 (특성에 따른 이원화)
memory = MemorySaver()

# 의도 분류용 (정밀 분석)
router_model = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 일반 질의용
general_model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

# 투자 질의용
invest_model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.2)



In [ ]:
# 3. 도구(Tools) 구성
# 일반 검색 도구
general_search = TavilySearch(
    topic="general",
    max_results=3,
    format_output=False,
)
general_search.name = "general_search"
general_search.description = (
    "뉴스 이외의 일반적인 주제나 지식에 대해 웹을 검색하려면 이 도구를 사용하세요."
)

# 투자/뉴스 검색 도구
invest_search = TavilySearch(
    topic="news",
    max_results=3,  # 최신 투자 동향을 위해 결과 수 확대
    format_output=False,
)
invest_search.name = "invest_search"
invest_search.description = "주식, 경제, 기업 정보, 최신 투자 뉴스 및 금융 시장 동향을 검색하려면 이 도구를 사용하세요."



In [ ]:
# 4. 의도 분류기(Intent Classifier) 정의
class IntentSchema(BaseModel):
    """사용자의 질문 의도를 분류하기 위한 스키마"""

    intent: str = Field(
        description="질문의 의도 종류. 일반적인 질문은 'general', 주식/부동산/가상자산/거시경제 등 투자 관련 질문은 'investment'를 반환합니다."
    )


# 구조화된 출력을 지원하는 분류기 생성
classifier = router_model.with_structured_output(IntentSchema)


def classify_intent(user_message: str) -> str:
    """사용자 메시지를 분석하여 'general' 또는 'investment'를 반환합니다."""
    system_instruction = "당신은 사용자의 질문을 분석하여 일반 질의(general)와 투자/경제 관련 질의(investment)로 완벽하게 분류하는 라우터입니다."

    response = classifier.invoke(
        [("system", system_instruction), ("human", user_message)]
    )
    return response.intent



In [ ]:
# 5. 실행 함수 (의도에 따른 에이전트 동적 생성 및 실행)
def run_agent_by_intent(user_query: str, thread_id: str):
    # 5-1. 의도 분류 수행
    intent = classify_intent(user_query)
    print(f"\n[SYSTEM] 감지된 의도 분류: {intent.upper()}")

    # 5-2. 의도에 따른 모델, 도구, 프롬프트 매핑
    if intent == "investment":
        selected_model = invest_model
        selected_tools = [invest_search]
        system_prompt = (
            "당신은 전문 수석 투자 분석가이자 금융 어시스턴트입니다. "
            "당신은 반드시 제공된 'invest_search' 도구를 활용하여 최신 시장 데이터를 확인한 뒤, "
            "당신은 거시경제(매크로) 시장 분석 전문가입니다. VIX, CLI, 환율, BDI, WTI, 장단기금리차 등 글로벌 경기 지표 등을 분석하여 현재 투자 환경을 평가하세요."
            "당신은 기업정보 분석 전문가입니다. 해당 기업의 비즈니스 모델, 재무제표(매출, 영업이익), 시장 점유율, 경쟁사 대비 강점을 분석하세요."
            "당신은 기술적 및 계량적 주가 분석 전문가입니다. 주가 추세, 차트 패턴, 거래량, 주요 이동평균선 및 밸류에이션(PER, PBR) 지표를 분석하세요."
            "당신은 경제 뉴스 분석 전문가입니다. 국내외 다양한 경제뉴스에서 투자 관점의 시장 이벤트를 찾아내어 분석하세요."
            "당신은 앞선 매크로 시장 분석, 기업정보 분석, 주가 분석, 뉴스 분석 등을 종합하여 국내외 마켓 상황 및 리스크 요인, 명확한 투자 의견(매수/매도/홀딩)이 포함된 '최종 투자전략 보고서'를 완성도 높게 작성하세요."
            "당신은 객관적인 사실과 수치에 기반하여 중립적이고 전문적인 투자 분석을 제공하세요. "
        )
    else:  # general
        selected_model = general_model
        selected_tools = [general_search]
        system_prompt = "당신은 항상 친절하고 간결하게 답변하는 AI 어시스턴트입니다."

    # 5-3. 조건에 맞는 React Agent 생성
    agent_executor = create_react_agent(
        selected_model,
        tools=selected_tools,
        checkpointer=memory,
        state_modifier=system_prompt,
    )

    # 5-4. 그래프 실행 및 스트리밍
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [("human", user_query)]}

    stream_graph(agent_executor, inputs, config, node_names=["agent", "tools"])
    


In [ ]:
# 6. 테스트 실행
# 테스트 1: 일반 질의 (gpt-4o-mini 작동)
run_agent_by_intent("북중미 월드컵 한국팀은 16강 가능할까?.", thread_id="abc123")

In [ ]:
# 테스트 2: 투자 관련 질의 (gpt-4o + 뉴스/금융 검색 작동)
run_agent_by_intent(
    "요즘 엔비디아(NVIDIA) 주가 흐름이랑 주요 실적 성장 요인이 뭔지 분석해줘.",
    thread_id="abc123",
)